In [16]:
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from dotenv import load_dotenv
import os
load_dotenv()
BASE_URL = "https://dev.to/api"
DEV_API_KEY = os.getenv("DEV_API_KEY")

# Set a user agent to avoid being blocked as a bot by the API edge.
DEFAULT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "CAP5771-DEV-Client/1.0 (contact: student@example.com)",
    "api-key": DEV_API_KEY,

}
EXCLUDE_TAGS = {
    "javascript",
    "react",
    "nextjs",
    "php",
    "html",
    "csharp",
    "angular",
    "ruby",
    "cryptocurrency",
    "bitcoin",
    "docker",
    "go",
    "codenewbie",
    "microservices",
    "flutter",
    "laravel",
    "kubernetes",
    "node",
    "typescript",
    "tutorial",
}


def fetch_articles(page=1, per_page=10, tag=None, username=None, headers=None):
    params = {
        "page": page,
        "per_page": per_page,
        "tags_exclude": ",".join(sorted(EXCLUDE_TAGS)),
    }

    url = f"{BASE_URL}/articles?{urlencode(params)}"
    request_headers = dict(DEFAULT_HEADERS)
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers)

    try:
        with urlopen(request) as response:
            payload = response.read().decode("utf-8")
            return json.loads(payload)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8")
        raise RuntimeError(f"DEV API error {exc.code}: {detail}") from exc

articles = fetch_articles(page=1, per_page=5, tag="python")

for article in articles:
    print(f"{article['title']} — {article['url']}")

Remember Your First Computer Book? — https://dev.to/richardpascoe/remember-your-first-computer-book-4fml
What was your win this week?? — https://dev.to/devteam/what-was-your-win-this-week-a3a
Metal Birds Watch: Copilot CLI Helped Me Watch Planes Without Looking Up — https://dev.to/georgekobaidze/metal-birds-watch-copilot-cli-helped-me-watch-planes-without-looking-up-4ha0
Symfony AI: A Rocket, a School Bus, or Something In Between? — https://dev.to/jeandevbr/symfony-ai-a-rocket-a-school-bus-or-something-in-between-31nn
Design a Movie Review Page — https://dev.to/richardpascoe/design-a-movie-review-page-36bl


In [17]:
import json
import re
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import time


def fetch_article_by_id(article_id, headers=None, retries=3):
    url = f"{BASE_URL}/articles/{article_id}"
    request_headers = dict(DEFAULT_HEADERS)
    if headers:
        request_headers.update(headers)

    for attempt in range(retries):
        try:
            request = Request(url, headers=request_headers)
            print(f"[FETCH] Attempt {attempt + 1}/{retries} for article {article_id}")
            with urlopen(request) as response:
                payload = response.read().decode("utf-8")
                return json.loads(payload)
        except HTTPError as exc:
            if exc.code == 404:
                print(f"[FETCH] Article {article_id} not found (404), skipping")
                return None
            if exc.code == 429:
                print(f"[FETCH] Rate limited (429), waiting 5s before retry...")
                time.sleep(5)
                continue
            detail = exc.read().decode("utf-8")
            print(f"[FETCH] HTTP error {exc.code}: {detail}")
            if attempt == retries - 1:
                return None
            time.sleep(1)
        except Exception as e:
            print(f"[FETCH] Error on attempt {attempt + 1}: {e}")
            if attempt == retries - 1:
                return None
            time.sleep(1)
    
    return None


def markdown_to_text(markdown):
    # Remove code blocks and inline code
    markdown = re.sub(r"```[\s\S]*?```", " ", markdown)
    markdown = re.sub(r"`[^`]*`", " ", markdown)

    # Remove images and links but keep link text
    markdown = re.sub(r"!\[[^\]]*\]\([^\)]*\)", " ", markdown)
    markdown = re.sub(r"\[([^\]]+)\]\([^\)]*\)", r"\1", markdown)

    # Remove HTML tags
    markdown = re.sub(r"<[^>]+>", " ", markdown)

    # Strip markdown formatting characters
    markdown = re.sub(r"[#>*_~\-]+", " ", markdown)

    # Normalize whitespace
    return " ".join(markdown.split())

In [ ]:
import sqlite3
import time
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

DB_PATH = "dev_ai_articles_full.sqlite"
TAGS = [None]  # Fetch all articles; server-side tags_exclude handles tag filtering.
PER_PAGE = 500
DAYS_BACK = 2300
MAX_WORKERS = 5
BATCH_SIZE = 100

KEYWORDS = [
    "hallucination",
    "hallucinations",
    "context window",
    "context length",
    "context limit",
    "context issues",
    "token limit",
    "long context",
    "prompt injection",
    "grounding",
    "retrieval",
    "rag",
    "automation",
    "automate",
    "agent",
    "autonomous",
    "workflow automation",
    "environmental impact",
    "carbon footprint",
    "energy usage",
    "energy consumption",
    "sustainability",
    "green ai",
    "layoff",
    "layoffs",
    "downsizing",
    "redundancy",
    "reduction in force",
    "rif",
    "bias",
    "fairness",
    "discrimination",
    "toxic",
    "harmful",
    "privacy",
    "data leakage",
    "pii",
    "data retention",
    "jailbreak",
    "jailbreaking",
    "accuracy",
    "factuality",
    "verification",
    "guardrails",
    "alignment",
    "safety",
    "copyright",
    "licensing",
    "plagiarism",
    "training data",
    "regulation",
    "policy",
    "compliance",
    "governance",
    "ai act",
    "job displacement",
    "reskilling",
    "productivity",
    "augmentation",
    "inference cost",
    "compute",
    "pricing",
    "vendor lock-in",
    "lock-in",
    "deepfake",
    "deepfakes",
    "misinformation",
    "disinformation",
    "fake news",
    "ai ethics",
]

total_failed_fetch = 0
total_keyword_filtered = 0


def parse_iso8601(value):
    if not value:
        return None
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(value)
    except ValueError:
        return None


def ensure_schema(connection):
    print("[DB] Creating schema if needed...")
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            url TEXT NOT NULL,
            published_at TEXT,
            tags TEXT,
            body_text TEXT,
            impressions INTEGER
        )
        """
    )
    connection.commit()
    print("[DB] Schema ready")


def upsert_article(connection, article):
    connection.execute(
        """
        INSERT INTO articles (id, title, url, published_at, tags, body_text, impressions)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id) DO UPDATE SET
            title=excluded.title,
            url=excluded.url,
            published_at=excluded.published_at,
            tags=excluded.tags,
            body_text=excluded.body_text,
            impressions=excluded.impressions
        """,
        (
            article["id"],
            article["title"],
            article["url"],
            article.get("published_at"),
            article.get("tags"),
            article.get("body_text"),
            article.get("impressions"),
        ),
    )


def is_ai_related(title, body_text):
    text = f"{title} {body_text}".lower()
    matched = {keyword for keyword in KEYWORDS if keyword in text}
    return len(matched) >= 3


def fetch_tagged_articles(tag, cutoff):
    page = 1
    article_count = 0
    tag_label = tag if tag else "(all)"
    while True:
        print(f"[API] Fetching page {page} for tag {tag_label}...")
        try:
            batch = fetch_articles(page=page, per_page=PER_PAGE, tag=tag)
        except Exception as e:
            print(f"[ERROR] Failed to fetch page {page}: {e}")
            break
            
        if not batch:
            print(f"[API] No more articles for tag {tag_label}")
            break

        published_dates = [parse_iso8601(item.get("published_at")) for item in batch]
        published_dates = [dt for dt in published_dates if dt]
        if published_dates:
            oldest = min(published_dates)
            newest = max(published_dates)
            print(f"[API] Page {page} date range: {oldest.date()} .. {newest.date()}")

        print(f"[API] Got {len(batch)} articles on page {page}")
        for item in batch:
            published = parse_iso8601(item.get("published_at"))
            if published and published < cutoff:
                print(f"[API] Reached cutoff date, stopping tag {tag_label}")
                return
            yield item
            article_count += 1

        page += 1
        print(f"[API] Sleeping 0.2s before next page...")
        time.sleep(0.2)
    
    print(f"[API] Total articles yielded for tag {tag_label}: {article_count}")


def fetch_and_process_article(article_id):
    """Fetch article body and process it. Returns record or None."""
    try:
        detail = fetch_article_by_id(article_id)
    except Exception as e:
        print(f"[ERROR] Failed to fetch article {article_id}: {e}")

        return None
    
    if detail is None:
        print(f"[SKIP] Skipping article {article_id} due to fetch failure")

        return None
        
    body_markdown = detail.get("body_markdown", "")
    body_text = markdown_to_text(body_markdown)
    title = detail.get("title", "")

    if not is_ai_related(title, body_text):
        print(f"[FILTER] Excluding article {article_id} (keyword mismatch)")
        return None
    
    record = {
        "id": article_id,
        "title": title,
        "url": detail.get("url", ""),
        "published_at": detail.get("published_at"),
        "tags": ",".join(detail.get("tag_list", [])),
        "body_text": body_text,
        "impressions": detail.get("impressions"),
    }
    print(f"[THREAD] Processed article {article_id}")
    return record


cutoff_date = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)
print(f"[MAIN] Cutoff date: {cutoff_date}")

connection = sqlite3.connect(DB_PATH)
ensure_schema(connection)

seen_ids = set()
total_fetched = 0
total_saved = 0

for tag in TAGS:
    print(f"\n[MAIN] Starting tag: {tag if tag else '(all)'}")
    articles_to_fetch = []
    
    # Collect article IDs first
    for item in fetch_tagged_articles(tag, cutoff_date):
        article_id = item["id"]
        if article_id in seen_ids:
            print(f"[DEDUP] Skipping duplicate article {article_id}")
            continue
        seen_ids.add(article_id)
        articles_to_fetch.append(article_id)
        total_fetched += 1
    
    # Fetch all articles concurrently
    print(f"[MAIN] Fetching {len(articles_to_fetch)} articles with {MAX_WORKERS} threads...")
    batch_records = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_and_process_article, aid): aid for aid in articles_to_fetch}
        
        for future in as_completed(futures):
            record = future.result()
            if record:
                batch_records.append(record)
                total_saved += 1
                
                # Insert batch when it reaches BATCH_SIZE
                if len(batch_records) >= BATCH_SIZE:
                    for rec in batch_records:
                        upsert_article(connection, rec)
                    connection.commit()
                    print(f"[DB] Batch committed ({total_saved} total articles saved)")
                    batch_records = []
    
    # Insert remaining records
    if batch_records:
        for rec in batch_records:
            upsert_article(connection, rec)
        connection.commit()
        print(f"[DB] Final batch committed ({total_saved} total articles saved)")

connection.close()

print(
    f"\n[MAIN] Complete. Fetched: {total_fetched}, Saved: {total_saved}, "
    f"Fetch failures: {total_failed_fetch}, Keyword filtered: {total_keyword_filtered}"
)

[MAIN] Cutoff date: 2026-01-25 18:47:11.878913+00:00
[DB] Creating schema if needed...
[DB] Schema ready

[MAIN] Starting tag: (all)
[API] Fetching page 1 for tag (all)...
[API] Page 1 date range: 2025-12-16 .. 2026-02-13
[API] Got 500 articles on page 1
[API] Reached cutoff date, stopping tag (all)
[MAIN] Fetching 105 articles with 5 threads...
[FETCH] Attempt 1/3 for article 3247048
[FETCH] Attempt 1/3 for article 3222108
[FETCH] Attempt 1/3 for article 3242019
[FETCH] Attempt 1/3 for article 3254071
[FETCH] Attempt 1/3 for article 3245863
[FILTER] Excluding article 3222108 (keyword mismatch)
[FETCH] Attempt 1/3 for article 3255059
[FILTER] Excluding article 3247048 (keyword mismatch)
[FETCH] Attempt 1/3 for article 3241863
[THREAD] Processed article 3242019
[FETCH] Attempt 1/3 for article 3255007
[THREAD] Processed article 3254071
[FETCH] Attempt 1/3 for article 3208282
[FILTER] Excluding article 3245863 (keyword mismatch)
[FETCH] Attempt 1/3 for article 3244205
[FILTER] Excluding a

In [19]:
import sqlite3

DB_PATH = "dev_ai_articles_full.sqlite"

with sqlite3.connect(DB_PATH) as conn:
    count = conn.execute("SELECT COUNT(*) FROM articles").fetchone()[0]
    print(f"Articles in DB: {count}")

Articles in DB: 232


In [20]:
import sqlite3
import pandas as pd

DB_PATH = "dev_ai_articles_full.sqlite"

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT id, title, url, published_at, tags, impressions, body_text FROM articles",
        conn,
    )

df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")

df

,id,title,url,published_at,tags,impressions,body_text
0,2917013,Building a Robust Classifier with Stacked Gene...,https://dev.to/ddebajyati/building-a-robust-cl...,2026-02-10 17:44:43+00:00,"p,y,t,h,o,n,,, ,d,a,t,a,s,c,i,e,n,c,e,,, ,m,a,...",None,I am thinking to start writing an ML series. T...
1,2948790,[Gemini][Python] Trying to Integrate a LINE Bo...,https://dev.to/gde/geminipython-trying-to-inte...,2026-01-11 15:15:49+00:00,"g,e,m,i,n,i,,, ,a,g,e,n,t,s,,, ,p,y,t,h,o,n,,,...",None,"Currently, there are many frameworks related t..."
2,3043958,When AI Says No,https://dev.to/rawveg/when-ai-says-no-4igo,2026-02-12 12:00:00+00:00,"h,u,m,a,n,i,n,t,h,e,l,o,o,p,,, ,a,i,s,h,u,t,d,...",None,"In a laboratory test conducted in July 2025, r..."
3,3079838,[Gemini][Google Maps] Building Location-Aware ...,https://dev.to/gde/geminigoogle-maps-building-...,2026-01-11 15:13:14+00:00,"g,e,m,i,n,i,,, ,g,o,o,g,l,e,,, ,a,p,i,,, ,a,i",None,"Background When developing a LINE Bot, I wante..."
4,3098356,Production-Ready AI with Google Cloud Learning...,https://dev.to/googleai/production-ready-ai-wi...,2026-01-20 14:51:04+00:00,"a,i,,, ,v,e,r,t,e,x,a,i,,, ,a,g,e,n,t,s,,, ,s,...",None,We're excited to launch the Production Ready A...
...,...,...,...,...,...,...,...
227,3250867,HTML Accessibility Review,https://dev.to/richardpascoe/html-accessibilit...,2026-02-12 14:00:13+00:00,"c,o,m,m,u,n,i,t,y,,, ,l,e,a,r,n,i,n,g,,, ,p,r,...",None,The HTML Accessibility review on freeCodeCamp ...
228,3251337,AI Won't Save You If You Don't Know What Good ...,https://dev.to/joietej/ai-wont-save-you-if-you...,2026-02-12 11:48:29+00:00,"a,i,,, ,p,r,o,g,r,a,m,m,i,n,g,,, ,s,o,f,t,w,a,...",None,Every other post on your feed is telling you A...
229,3254037,How to Detect Prompt Injection Attacks in Your...,https://dev.to/zeshama/how-to-detect-prompt-in...,2026-02-13 15:27:58+00:00,"a,i,,, ,s,e,c,u,r,i,t,y,,, ,t,y,p,e,s,c,r,i,p,...",None,Your AI agent accepts user input. That means s...
230,3254069,The 4 Rules of Simple Design: A Practical Guid...,https://dev.to/maximeshr/the-4-rules-of-simple...,2026-02-13 12:14:01+00:00,"p,r,o,g,r,a,m,m,i,n,g,,, ,t,y,p,e,s,c,r,i,p,t,...",None,Kent Beck introduced the 4 Rules of Simple Des...
